### LIBRARY IMPORTS

In [1]:
import pandas as pd
import numpy as np
import os
import torch
import torch.nn as nn
import torch.optim as optim
import copy


from sklearn.metrics import accuracy_score

if os.getcwd().endswith('notebooks'):
    os.chdir('..')

from src import *
from src.data_manager import DataManager
from src.processor import Processor
from src.cnn_regressor import CNNRegressor

### CONFIGURATION

In [2]:
data_manager = DataManager()

datasets_config, modeling_config = data_manager.load_config()

active_dataset = "cifar10"
active_dataset_config = datasets_config[active_dataset]

problem_type = active_dataset_config["problem_type"]

gradient_boosting_config = modeling_config["gradient_boosting"]

weak_learner_key = gradient_boosting_config["weak_learner_key"]
weak_learner_config = modeling_config[weak_learner_key]

processor = Processor(**active_dataset_config)

train, valid, test = data_manager.load_image_data(
    active_dataset
)

### ACCURACY ON CONVOLUTIONAL GRADIENT BOOSTING

In [3]:
preds_dir = f"{MODELS_PATH}/{active_dataset}/2026_03_24_12_25/predictions.csv"
preds = pd.read_csv(preds_dir).astype(int)[active_dataset_config["target"]].values

y_test = np.array(test.dataset.targets)[test.indices]

accuracy_score(y_test, preds)

0.7035

### SMALL CNN

In [4]:
class SmallCNN(CNNRegressor):
    def __init__(self):
        super().__init__(epochs=100, channels=8, kernel_size=5, pool_size=2, hidden_size=16, batch_size=64)

    def fit(self, X_train, y_train, X_valid, y_valid, patience=10):
        
        output_size = int(y_train.max() + 1)
        in_channels = X_train.shape[1]

        image_size = X_train.shape[-1]
        conv1_out = (image_size - (self.kernel_size - 1)) / self.pool_size
        conv2_out = (conv1_out - (self.kernel_size - 1)) / self.pool_size
        linear_input = self.channels * conv2_out ** 2
        
        self._get_network(in_channels, int(linear_input), output_size)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters())
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch {epoch + 1}: Validation Log Loss {val_loss:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

small_cnn = SmallCNN()
small_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

test_numpy = processor.convert_to_numpy(test) 
raw_preds = small_cnn.predict(test_numpy) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch 1: Validation Log Loss 1.6647
Epoch 2: Validation Log Loss 1.5596
Epoch 3: Validation Log Loss 1.4986
Epoch 4: Validation Log Loss 1.4492
Epoch 5: Validation Log Loss 1.4470
Epoch 6: Validation Log Loss 1.3848
Epoch 7: Validation Log Loss 1.3981
Epoch 8: Validation Log Loss 1.3430
Epoch 9: Validation Log Loss 1.3299
Epoch 10: Validation Log Loss 1.3491
Epoch 11: Validation Log Loss 1.3217
Epoch 12: Validation Log Loss 1.3164
Epoch 13: Validation Log Loss 1.3065
Epoch 14: Validation Log Loss 1.2848
Epoch 15: Validation Log Loss 1.3151
Epoch 16: Validation Log Loss 1.2940
Epoch 17: Validation Log Loss 1.2814
Epoch 18: Validation Log Loss 1.2706
Epoch 19: Validation Log Loss 1.2571
Epoch 20: Validation Log Loss 1.2601
Epoch 21: Validation Log Loss 1.2573
Epoch 22: Validation Log Loss 1.2668
Epoch 23: Validation Log Loss 1.2415
Epoch 24: Validation Log Loss 1.2336
Epoch 25: Validation Log Loss 1.2346
Epoch 26: Validation Log Loss 1.2583
Epoch 27: Validation Log Loss 1.2502
Epoch 28: 

In [6]:
class BigCNN(CNNRegressor):
    def __init__(self):
        super().__init__(epochs=100, channels=32, kernel_size=5, pool_size=2, hidden_size=32, batch_size=32)

    def fit(self, X_train, y_train, X_valid, y_valid, patience=10):
        
        output_size = int(y_train.max() + 1)
        in_channels = X_train.shape[1]

        image_size = X_train.shape[-1]
        conv1_out = (image_size - (self.kernel_size - 1)) / self.pool_size
        conv2_out = (conv1_out - (self.kernel_size - 1)) / self.pool_size
        linear_input = self.channels * conv2_out ** 2
        
        self._get_network(in_channels, int(linear_input), output_size)

        train_loader = self._prepare_loader(X_train, y_train)
        
        X_valid_t = torch.from_numpy(X_valid).to(torch.float32).to(self.device)
        y_valid_t = torch.from_numpy(y_valid).long().to(self.device).view(-1)
        
        criterion = nn.CrossEntropyLoss()
        optimizer = optim.Adam(self.parameters())
        
        best_loss = float('inf')
        best_model = None
        early_stop_count = 0

        for epoch in range(self.epochs):
            self.train()
            for batch_X, batch_y in train_loader:
                batch_X, batch_y = batch_X.to(self.device), batch_y.to(self.device)
                
                optimizer.zero_grad()
                preds = self(batch_X)
                
                loss = criterion(preds, batch_y.long().view(-1))
                loss.backward()
                optimizer.step()

            self.eval()
            with torch.no_grad():
                val_preds = self(X_valid_t)
                val_loss = criterion(val_preds, y_valid_t).item()

            if (epoch + 1) % 1 == 0:
                print(f"Epoch {epoch + 1}: Validation Log Loss {val_loss:.4f}")

            if val_loss < best_loss:
                best_loss = val_loss
                best_model = copy.deepcopy(self.state_dict())
                early_stop_count = 0
            else:
                early_stop_count += 1

            if early_stop_count >= patience:
                break
                
        if best_model:
            self.load_state_dict(best_model)

X_train, y_train = processor.split_features_target(train)
X_valid, y_valid = processor.split_features_target(valid)

big_cnn = BigCNN()
big_cnn.fit(X_train, y_train, X_valid, y_valid)

print ("-" * 50)

test_numpy = processor.convert_to_numpy(test) 
raw_preds = big_cnn.predict(test_numpy) 

if len(raw_preds.shape) > 1 and raw_preds.shape[1] > 1:
    preds = np.argmax(raw_preds, axis=1)
else:
    preds = raw_preds.flatten().astype(int)

score = accuracy_score(y_test, preds)
print(f"Test Accuracy: {score:.4f}")

Epoch 1: Validation Log Loss 1.2920
Epoch 2: Validation Log Loss 1.1503
Epoch 3: Validation Log Loss 1.1330
Epoch 4: Validation Log Loss 1.0732
Epoch 5: Validation Log Loss 1.0660
Epoch 6: Validation Log Loss 1.0637
Epoch 7: Validation Log Loss 1.0393
Epoch 8: Validation Log Loss 1.0656
Epoch 9: Validation Log Loss 1.0578
Epoch 10: Validation Log Loss 1.0883
Epoch 11: Validation Log Loss 1.0601
Epoch 12: Validation Log Loss 1.0805
Epoch 13: Validation Log Loss 1.0662
Epoch 14: Validation Log Loss 1.1110
Epoch 15: Validation Log Loss 1.1348
Epoch 16: Validation Log Loss 1.1321
Epoch 17: Validation Log Loss 1.1784
--------------------------------------------------
Test Accuracy: 0.6416
